# 已保存尾部概率的适当评分
比较同定义SOXX尾部概率，不重新训练、推理或回测。待独立评审。

In [1]:
from pathlib import Path
import sys, qlib
root=Path.cwd()
while not (root/"SPEC.md").exists(): root=root.parent
sys.path.insert(0,str(root))
qlib.init(provider_uri=str(root/"data/qlib/frozen_policy_raw_cashbook_20261015"),region="us",kernels=1)
from src.workflow.generic_index_tail_scoring import run
result=run(root)
result

[1488:MainThread](2026-10-15 23:01:41,021) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


2026/10/15 23:01:41 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[1488:MainThread](2026-10-15 23:01:41,518) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[1488:MainThread](2026-10-15 23:01:41,518) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/frozen_policy_raw_cashbook_20261015')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


...

[1488:MainThread](2026-10-15 23:01:43,745) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name generic_index_tail_scoring_20261015.


[1488:MainThread](2026-10-15 23:01:43,747) INFO - qlib.workflow - [exp.py:258] - Experiment 832350012044339925 starts running ...


[1488:MainThread](2026-10-15 23:01:43,831) INFO - qlib.workflow - [recorder.py:345] - Recorder 85d314b1f0ac470e856da416a2f156cd starts running under Experiment 832350012044339925 ...


[1488:MainThread](2026-10-15 23:01:45,073) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{'experiment': 'generic_index_tail_scoring_20261015',
 'plan_sha256': 'ae4ef830a9d8e9995c495c6a12db0dcb8cf46109d2fb502557c44f81324e29f8',
 'decision': 'no_reliable_saved_tail_forecast_improvement',
 'primary_logloss_ci': {'paired_mean_loss_improvement': -0.09125544730360703,
  'lower_bound': -0.1475460873114306,
  'upper_bound': -0.04196168198506539,
  'valid_replicates': 2000,
  'invalid_replicates': 0,
  'calendar_rows': 944,
  'scored_rows': 940},
 'calendar_rows': 944,
 'scored_rows': 940,
 'unknown_rows': 4,
 'event_count': 105,
 'last_scored_signal': '2026-10-01 00:00:00',
 'last_used_maturity': '2026-10-09 09:31:00-04:00',
 'parent_asof': '2026-10-09 09:47:00-04:00',
 'parent_portfolio_route_remains_closed': True,
 'new_inference': 0,
 'new_fits': 0,
 'new_policy_accounts': 0,
 'new_fetches': 0,
 'objective_achieved': False,
 'recorder_id': '85d314b1f0ac470e856da416a2f156cd',
 'experiment_id': '832350012044339925',
 'recorder_uri': '/Users/rziiiii/dev/workspace/research/generic_

# 指数辅助尾部概率：评分变差

直接读取已保存的控制/候选三种子平均尾部概率和原始SOXX 1x H5入口相对路径标签，没有新增推理、训练、阈值、校准或策略回放。保留全部944个日历位置；按父试验2026-10-09 09:47纽约时点，940个标签成熟、4个末端标签未知。最后可评分信号2026-10-01，最后成熟时点2026-10-09 09:31；不使用当前日期补回未来结果。

主要配对指标为“控制logloss−候选logloss”，正值才代表候选改善。结果 **−0.09125545**，固定63日依赖区块、2000次重采样的2.5%/97.5%分位区间为 **[−0.14754609, −0.04196168]**；全部2000次可估计。这里下界对应单侧97.5%规则。候选没有改善原始尾部预测，且这个条件区间整体位于零以下。

| 指标 | 无辅助控制 | 指数尾部辅助候选 |
|---|---:|---:|
| logloss（越低越好） | 0.343716 | 0.434972 |
| Brier（描述性，越低越好） | 0.098389 | 0.133975 |
| 可评分日平均尾部概率 | 14.0324% | 27.4704% |
| 真实尾部频率 | 11.1702% | 11.1702% |
| 非事件加权logloss贡献 | 0.133456 | 0.301354 |
| 事件加权logloss贡献 | 0.210260 | 0.133618 |

105个事件上，候选给出的概率更高，事件损失下降；但835个非事件上的损失增加更多，主评分总体恶化。2023、2024、2025、2026各信号年度的配对logloss改善分别为−0.086927、−0.057118、−0.064187、−0.178765，全部为负；首末年为边界部分年度。所有季度和事件/非事件单元均保留，不筛选有利区间。固定1e−7数值裁剪实际触发0行；Brier使用原始概率。

**结论：no_reliable_saved_tail_forecast_improvement。** 这说明父试验的历史组合点估计改善不能被解释为“原始尾部概率预测变准”。概率水平明显上移与非事件损失上升可直接观察，但不能据此认定某一梯度、训练基准率或控制器是唯一因果原因。也不意味着所有通用市场迁移机制都失败。

父组合试验仍为不推广：滚动稳定性、强参考与置信区间门失败。关闭这套固定指数尾部辅助路线，不进行概率校准、辅助权重、阈值或模型救援。下一步若继续，必须是去重后的不同ETF预测机制和新冻结计划；本评分不授权新训练，也不引入个股方向。

原始当前版本历史标签、重叠H5事件和重用评估历史限制仍在；区块区间不涵盖模型选择/重训不确定性，也不是独立前向证据。共享成熟掩码、概率范围、二元标签及事件贡献加和均在执行notebook中断言，复用既有完整日历bootstrap实现，没有增加测试框架。

Recorder `85d314b1f0ac470e856da416a2f156cd`；NAS `ac3bfdf8aa3e7dcb1bf7a7cb883e906801532e53bed3c15feaa3a9f3bd818996`。全部944行、成熟掩码、评分及重采样索引/结果保存在Recorder，独立评审和冷读由分离角色完成。

独立最终评估已复算并接受评分恶化结论；负责人NAS冷恢复核验20个对象哈希一致。原组合路线保持关闭，不追加校准或阈值试验。
